# R125 資料與模型契約核對

## tl;dr

**第二版的 64 場景／128 個觀測全部通過原先門檻，只准入已知模擬條件下的有限訓練小試驗。** 第一版的拒絕和所有資料保持原狀。最差觀測 RMS／σ 從 **0.817690** 降至 **0.166013**，固定上限為 0.25；最差觀測 p99／σ 從 **2.047052** 降至 **0.457476**，固定上限為 1。

改善來自提高光學強度表示的精細度和幾何計算精度。所有 64 個場景的形狀、亮暗、光學、雜訊及位移條件均保持相同，沒有移除失敗案例。真實儀器校準與真實效益尚未由這項檢查證明。

## Context & Methods

這是一份技術讀者用的資料品質核對筆記本。決策是：新程序資料是否足以進入**已知模擬成像條件下的有限訓練小試驗**。真實顯微影像與 SEM 的模型效益另行判定。本筆記本只讀取完成的核對 JSON、清單與中繼資料；重新計算母數、最大值、失敗數、分層與來源一致性，不重新生成資料、不載入模型、不使用 GPU。

### Key Assumptions

- 每版保留相同 64 個概念場景，每場景兩個觀測；每張原生影像為 640×640，監督區域是 `[256:384,256:384]` 的 128×128 中央區域。
- 對每個觀測，以已存理想光學強度經模型成像計算的结果減去獨立生成的無雜訊觀測。RMS 與絕對誤差第 99 百分位分別除以 `sqrt(profile.noise_variance)`。
- `noise_variance = (max(actual_noise_sigma_gray,0.5)^2 + 1/12)/255^2`。其中 0.5 是數值下限，1/12 是量化變異的近似；不能說原樣保持案例真的加入了這些雜訊。
- 固定門檻：每個觀測 RMS≤0.25σ、p99≤1σ；理想光學強度重新積分與原生目標最大差≤2e-7。這些是工程入場條件，沒有變成真實儀器適用性定理。
- 下文「最差」是 128 個觀測各自統計的最大值；不是把各觀測的 p99 再稱作所有像素合併後的 p99。


原始成像算子核對的運算環境為 v1 CPU／Float32、v2 GPU／Float32；新版 64/64 筆記錄為 `cuda`。8×Float64 指資料形成與獨立物理重算，並非模型運算精度。兩版數字比較是整套修正後的入場核對，不是隔離每項改動效應的模型效能消融。本筆記本只在 CPU 重整已存數值。

## Data

### 1. 匯入與固定來源

執行環境使用 `C:/Users/Rio/anaconda3/python.exe`，需要 pandas、nbformat、nbclient 與 ipykernel。來源路徑與已存指紋在下面完整列出。

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib, json, platform
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path(r"..")
pd.set_option("display.max_rows", 16)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 68)
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
audit_v1 = read_json(ROOT / "results/data_audit_v1.json")
audit_v2 = read_json(ROOT / "results/data_audit_v2.json")
manifest_v1 = read_json(ROOT / "data/package/manifest.json")
manifest_v2 = read_json(ROOT / "data/package_v2/manifest.json")
precision = read_json(ROOT / "results/data_audit_scene007_precision_diagnosis.json")
decomposition = read_json(ROOT / "results/data_audit_representation_decomposition.json")
integrity = read_json(ROOT / "results/data_audit_v2_integrity_review.json")
print({"python": platform.python_version(), "pandas": pd.__version__, "audit_v2_utc": audit_v2["utc"], "scope": audit_v2["scope"]})

{'python': '3.12.7', 'pandas': '2.2.2', 'audit_v2_utc': '2026-09-08T07:24:23.728651+00:00', 'scope': 'known-simulation bounded pipeline only'}


### 2. 核對來源身分與原判準

舊版拒絕與新版通過分別綁定自己的資料清單；舊清單、舊判準和失敗紀錄沒有被重新接受成新版證據。以下重新比對已存的指紋。

In [2]:
sources = [
    ("v1 package manifest", ROOT / "data/package/manifest.json", audit_v1["manifest_sha256"]),
    ("v2 package manifest", ROOT / "data/package_v2/manifest.json", audit_v2["manifest_sha256"]),
    ("v1 generator", ROOT / "data/generator.py", manifest_v1["generator_sha256"]),
    ("v2 generator", ROOT / "data/generator_v2.py", audit_v2["source_generator_sha256"]),
    ("v1 admission contract", ROOT / "config/data_admission.json", audit_v1["admission_contract_sha256"]),
    ("v2 admission contract", ROOT / "config/data_admission_v2.json", audit_v2["data_admission_contract_sha256"]),
    ("v2 audit report", ROOT / "results/data_audit_v2.json", integrity["v2_report_sha256"]),
]
source_rows = []
for label, path, expected in sources:
    actual = sha256(path)
    assert actual == expected, (label, "source changed")
    source_rows.append({"source": label, "path": str(path), "sha256": actual, "matches_saved_identity": True})
assert audit_v1["screen_limits_fixed_before_measurement"] == audit_v2["unchanged_limits"]
assert audit_v1["pilot_admitted"] is False
assert audit_v2["pilot_admitted"] is True
assert integrity["old_scene_pairs_rechecked"] == 64
assert integrity["all_v1_payloads_and_metadata_unchanged"] is True
source_table = pd.DataFrame(source_rows)
display(source_table[["source", "path", "matches_saved_identity"]])
display(pd.DataFrame([audit_v1["screen_limits_fixed_before_measurement"], audit_v2["unchanged_limits"]], index=["v1", "v2"]))

,source,path,matches_saved_identity
0,v1 package manifest,../data/package/ma...,True
1,v2 package manifest,../data/package_v2...,True
2,v1 generator,../data/generator.py,True
3,v2 generator,../data/generator_...,True
4,v1 admission contract,../config/data_adm...,True
5,v2 admission contract,../config/data_adm...,True
6,v2 audit report,../results/data_au...,True


,target_area_max_abs,operator_discrepancy_rms_over_declared_sigma,operator_discrepancy_p99_over_declared_sigma
v1,2.000000e-07,0.25,1.0
v2,2.000000e-07,0.25,1.0


### 3. 為何舊資料不能直接接入

這是有限範圍的介面與來源核對：R118 清單及首筆四個實際陣列、R120 銀行清單與載入方式、R61 啟動設定。不是對所有舊樣本逐筆宣布無效，也不是從教師或 SEM 標準答案的存在推定作弊。

In [3]:
legacy = audit_v1["legacy"]
legacy_paths = {
    "R118": ROOT.parent / "R118_PhysicalCoordinateRepairScratch/data/package/manifest.json",
    "R120": ROOT.parent / "R120_PhysicalSequenceRepair/data/sequences/train/manifest.json",
    "R61": ROOT.parent.parent / "Thesis_MOSR_Evidence/LabSystem_MOSR/Experiments/R61_BoundaryGeneralizationUniversal/scripts/launch_r61.py",
}
expected_legacy_hashes = {
    "R118": legacy["R118"]["manifest_sha256"],
    "R120": legacy["R120"]["bank_manifest_sha256"],
    "R61": legacy["R61"]["launcher_sha256"],
}
for name, path in legacy_paths.items():
    assert sha256(path) == expected_legacy_hashes[name], (name, "legacy source changed")
legacy_rows = [
    {"source": "R61", "observed": "256px default; SEM truth role and declared old-model teacher", "direct_use": False},
    {"source": "R118", "observed": f"{legacy['R118']['records']} records; first pair 256px, different PSFs, no declared fine-grid latent target", "direct_use": False},
    {"source": "R120", "observed": f"physical bank {legacy['R120']['physical_bank_scenes']}/{legacy['R120']['base_probe_scenes']} scene IDs; remaining loader fallback", "direct_use": False},
]
display(pd.DataFrame(legacy_rows))
print("R120 scene-ID coverage:", legacy["R120"]["physical_bank_scenes"] / legacy["R120"]["base_probe_scenes"])
print("Inspection scope:", legacy["scope"])

,source,observed,direct_use
0,R61,256px default; SEM truth role and declared old-model teacher,False
1,R118,"5512 records; first pair 256px, different PSFs, no declared fine...",False
2,R120,physical bank 96/512 scene IDs; remaining loader fallback,False


R120 scene-ID coverage: 0.1875
Inspection scope: R118 manifest and one actual four-array record; R120 bank manifest/loader; R61 launcher, not whole historical datasets


### 4. 重新點算場景、條件與實際中央內容

讀取兩版的 128 份小型場景說明，不載入大型影像陣列。形狀、亮暗、雜訊、光學條件和位移逐項比較；不以總場景數相同代替逐項一致。

In [4]:
metadata_v1 = {entry["scene_id"]: read_json(ROOT / "data/package" / entry["metadata_path"]) for entry in manifest_v1["scenes"]}
metadata_v2 = {entry["scene_id"]: read_json(ROOT / "data/package_v2" / entry["metadata_path"]) for entry in manifest_v2["scenes"]}
assert len(metadata_v1) == len(metadata_v2) == 64
assert metadata_v1.keys() == metadata_v2.keys()
condition_keys = ["index", "category", "seed", "effective_seed", "attempt", "shape_params", "photometry", "polarity", "shift_xy", "view_shift_xy", "identity", "flat", "noise_sigma_gray", "noise_variance_floor_gray", "source_ancestors"]
condition_rows = []
scene_rows = []
for scene_id, new in metadata_v2.items():
    old = metadata_v1[scene_id]
    equal = all(old[key] == new[key] for key in condition_keys)
    equal &= all(old["rendering_profile"][key] == new["rendering_profile"][key] for key in ["sigma_native_xy", "rotation_rad", "radius_native"])
    equal &= all(old["profile"][key] == new["profile"][key] for key in ["gain", "bias", "noise_variance"])
    assert equal, scene_id
    assert new["attempt"] == 0
    assert new["external_training_inputs"] == []
    assert all(stat["source_saturated_pixels"] == 0 for stat in new["core_stats"])
    condition_rows.append({"scene_id": scene_id, "physical_conditions_unchanged": bool(equal)})
    scene_rows.append({"scene_id": scene_id, "category": new["category"], "polarity": new["polarity"], "noise_sigma_gray": new["noise_sigma_gray"], "identity": new["identity"], "flat": new["flat"], "motion_observable": new["motion_observable"], "texture_core_std": min(stat["intrinsic_texture_std"] for stat in new["core_stats"])})
scenes = pd.DataFrame(scene_rows)
category_table = scenes.groupby("category", sort=False).size().rename("scenes").reset_index()
category_table["views"] = category_table["scenes"] * 2
assert category_table["scenes"].sum() == 64
assert category_table["views"].sum() == 128
display(category_table)
display(scenes.groupby("noise_sigma_gray").size().rename("scenes").to_frame())
print({"unchanged_conditions": len(condition_rows), "foreground_bright": int((scenes.polarity == 1).sum()), "foreground_dark": int((scenes.polarity == -1).sum()), "no_core_saturation": True})

,category,scenes,views
0,irregular_curved,13,26
1,corners_lines_junctions,13,26
2,dense_close_structures,13,26
3,textured_edges,13,26
4,clean_identity,6,12
5,flat_negative,6,12


,scenes
noise_sigma_gray,
0.0,6
0.5,15
1.5,22
3.0,21


{'unchanged_conditions': 64, 'foreground_bright': 34, 'foreground_dark': 30, 'no_core_saturation': True}


## Results

### 5. 以固定母數重新計算最差值與失敗數

每版共 128 個觀測、每個觀測 16,384 個中央像素，合計 2,097,152 個像素誤差。表內最大 p99 是「最差觀測的 p99」。絕對最大誤差另外保留，沒有因為低於一個灰階就省略。

In [5]:
def view_table(audit, version):
    rows = []
    for scene in audit["scenes"]:
        for view, values in enumerate(scene["operator_fit_per_view"]):
            rows.append({"version": version, "scene_id": scene["scene_id"], "view": view, "category": scene["category"], **values})
    frame = pd.DataFrame(rows)
    assert len(frame) == 128
    assert frame[["scene_id", "view"]].drop_duplicates().shape[0] == 128
    assert (frame["count"] == 16384).all()
    frame["operator_screen_failed"] = (frame.rms_over_declared_sigma > 0.25) | (frame.p99_abs_over_declared_sigma > 1.0)
    return frame
views_v1 = view_table(audit_v1, "v1")
views_v2 = view_table(audit_v2, "v2")
metric_rows = []
for version, frame, audit in [("v1", views_v1, audit_v1), ("v2", views_v2, audit_v2)]:
    metric_rows.append({"version": version, "views": len(frame), "core_pixels": int(frame["count"].sum()), "worst_RMS_over_sigma": float(frame.rms_over_declared_sigma.max()), "worst_p99_over_sigma": float(frame.p99_abs_over_declared_sigma.max()), "absolute_max_gray": float(frame.max_abs_gray.max()), "failed_operator_views": int(frame.operator_screen_failed.sum()), "failed_operator_scenes": int(frame.loc[frame.operator_screen_failed, "scene_id"].nunique()), "all_gate_failed_scenes": sum(bool(scene["failures"]) for scene in audit["scenes"]), "pilot_admitted": audit["pilot_admitted"]})
metrics = pd.DataFrame(metric_rows)
assert metrics.core_pixels.eq(2097152).all()
display(metrics)
print("V2 native target integration maximum:", max(scene["target_area_integration"]["max_abs"] for scene in audit_v2["scenes"]))

,version,views,core_pixels,worst_RMS_over_sigma,worst_p99_over_sigma,absolute_max_gray,failed_operator_views,failed_operator_scenes,all_gate_failed_scenes,pilot_admitted
0,v1,128,2097152,0.817690,2.047052,1.986290,6,3,4,False
1,v2,128,2097152,0.166013,0.457476,0.445655,0,0,0,True


V2 native target integration maximum: 4.842877388000488e-08


### 6. 保留原始失敗並逐個比較

007 的問題來自獨立高精度重算，不是成像誤差門檻；013、020、050 則違反模型成像與獨立生成之間的誤差門檻。四個案例都保留，沒有選擇性刪除。

In [6]:
failure_table = pd.DataFrame([{"scene_id": scene["scene_id"], "category": scene["category"], "preserved_v1_failure": "; ".join(scene["failures"])} for scene in audit_v1["scenes"] if scene["failures"]])
display(failure_table)
hard_ids = ["scene_007", "scene_013", "scene_020", "scene_050"]
columns = ["rms_over_declared_sigma", "p99_abs_over_declared_sigma", "max_abs_gray"]
old_hard = views_v1[views_v1.scene_id.isin(hard_ids)].groupby("scene_id")[columns].max().add_prefix("v1_")
new_hard = views_v2[views_v2.scene_id.isin(hard_ids)].groupby("scene_id")[columns].max().add_prefix("v2_")
hard_cases = old_hard.join(new_hard, validate="one_to_one").reset_index()
assert len(hard_cases) == 4
display(hard_cases.round(6))

,scene_id,category,preserved_v1_failure
0,scene_007,corners_lines_junctions,independent_float64_physical_patch_view1_differs_materially
1,scene_013,corners_lines_junctions,2x_model_vs_4x_render_discrepancy_exceeds_prespecified_screen
2,scene_020,dense_close_structures,2x_model_vs_4x_render_discrepancy_exceeds_prespecified_screen
3,scene_050,dense_close_structures,2x_model_vs_4x_render_discrepancy_exceeds_prespecified_screen


,scene_id,v1_rms_over_declared_sigma,v1_p99_abs_over_declared_sigma,v1_max_abs_gray,v2_rms_over_declared_sigma,v2_p99_abs_over_declared_sigma,v2_max_abs_gray
0,scene_007,0.039757,0.220244,0.361953,0.004157,0.021986,0.039898
1,scene_013,0.265381,1.585076,1.655997,0.059888,0.365435,0.377381
2,scene_020,0.817690,2.047052,1.706443,0.166013,0.457476,0.400871
3,scene_050,0.314676,1.231158,1.057711,0.053131,0.223554,0.215585


### 7. 分開「光學離散方式」與「已丟失的格內資訊」

插入同一個 4×物理成像計算，但把真實 4×強度替換為 2×平均值的重複。如此可以觀察粗格子平均化本身的資訊損失。各項是誤差影像的分解，RMS 不可直接相加；兩種錯誤可能部分抵消。因此只調整模糊核，未必改善問題。

In [7]:
decomposition_rows = []
for scene in decomposition["scenes"]:
    assert scene["decomposition_residual_max"] == 0
    decomposition_rows.append({"scene_id": scene["scene"], "observed_total_RMS_gray": scene["original_2x_vs_physical4x"]["rms_gray"], "lost_within_cell_RMS_gray": scene["physical4x_repeated2x_vs_true4x_basis_loss"]["rms_gray"], "kernel_discretization_RMS_gray": scene["sampled2x_kernel_vs_exact_repeated_cell_operator"]["rms_gray"]})
decomposition_table = pd.DataFrame(decomposition_rows)
display(decomposition_table.round(6))
assert decomposition["status"] == "diagnostic_no_data_or_kernel_changed"

,scene_id,observed_total_RMS_gray,lost_within_cell_RMS_gray,kernel_discretization_RMS_gray
0,scene_013,0.153117,0.241190,0.170799
1,scene_020,0.471959,0.687639,0.522495
2,scene_050,0.181664,0.479795,0.448236


### 8. 幾何精度與獨立局部重算

007 的已知位置在低精度運算中位於形狀內，在雙精度運算中位於形狀外。只差一個細格，仍足以改變附近模糊後的觀測。新版使用雙精度形成形狀與紋理，並由獨立程式重算 16 個局部區域。

In [8]:
precision_table = pd.DataFrame([{"physical_x": item["physical_xy"][0], "physical_y": item["physical_xy"][1], "float32_score": item["score_float32"], "float64_score": item["score_float64"], "inside_float32": item["inside_generator32"], "inside_independent64": item["inside_independent64"]} for item in precision["locations"]])
display(precision_table)
replay_rows = []
for scene in audit_v2["scenes"]:
    for view in (scene.get("independent8x_physical_replay") or {}).get("views", []):
        replay_rows.append({"scene_id": scene["scene_id"], "view": view["view"], "pixels": view["noiseless_error"]["count"], "noiseless_max_gray": view["noiseless_error"]["max_abs_gray"], "target_max_gray": view["target_error"]["max_abs_gray"]})
replays = pd.DataFrame(replay_rows)
assert len(replays) == 16
assert replays["pixels"].eq(256).all()
replay_summary = pd.DataFrame([{"patches": len(replays), "patch_pixels_total": int(replays["pixels"].sum()), "maximum_noiseless_error_gray": float(replays.noiseless_max_gray.max()), "maximum_target_error_gray": float(replays.target_max_gray.max()), "v1_scene007_local_max_error_gray": precision["float64_local_replay_vs_stored"]["max_abs_gray"]}])
display(replay_summary)

,physical_x,physical_y,float32_score,float64_score,inside_float32,inside_independent64
0,315.625,316.125,0.000001,-0.000003,True,False


,patches,patch_pixels_total,maximum_noiseless_error_gray,maximum_target_error_gray,v1_scene007_local_max_error_gray
0,16,4096,0.000008,0.000008,0.354192


### 9. 原樣保持與平坦背景是不同的控制

原樣保持有邊界、零位移、零實際雜訊及恆等光學；六個平坦背景可以有已指定的移動與雜訊，但沒有可觀察的形狀位移。兩者不混成「模型量對了位移」。

In [9]:
identity_ids = set(scenes.loc[scenes.identity, "scene_id"])
flat_ids = set(scenes.loc[scenes.flat, "scene_id"])
assert len(identity_ids) == len(flat_ids) == 6
assert all(scene["identity_exact"] is True for scene in audit_v2["scenes"] if scene["scene_id"] in identity_ids)
assert all(metadata_v2[scene_id]["exact_delta_xy"] == [0.0, 0.0] and metadata_v2[scene_id]["noise_sigma_gray"] == 0 for scene_id in identity_ids)
assert all(metadata_v2[scene_id]["motion_observable"] is False for scene_id in flat_ids)
control_table = pd.DataFrame([
    {"control": "clean_identity", "scenes": 6, "views": 12, "actual_noise": "0", "motion": "zero-motion control", "verified": "source/255 == target exactly"},
    {"control": "flat_negative", "scenes": 6, "views": 12, "actual_noise": "declared per scene", "motion": "unobservable", "verified": "no foreground boundary in core"},
])
display(control_table)
print({"all_conditions_preserved": all(row["physical_conditions_unchanged"] for row in condition_rows), "protected_sources_unchanged": audit_v2["protected_sources_unchanged"], "real_calibration_qualified": audit_v2["real_calibration_qualified"], "real_usefulness_established": audit_v2["real_usefulness_established"]})

,control,scenes,views,actual_noise,motion,verified
0,clean_identity,6,12,0,zero-motion control,source/255 == target exactly
1,flat_negative,6,12,declared per scene,unobservable,no foreground boundary in core


{'all_conditions_preserved': True, 'protected_sources_unchanged': True, 'real_calibration_qualified': False, 'real_usefulness_established': False}


## Takeaways

新版把獨立物理生成提高至 8×雙精度、模型光學強度提高至 4×；學習網路仍在 2×運作，但每個位置能提出四個獨立細格修正。這是提高表示能力與修正算術精度，沒有擴大允許誤差、增加雜訊、變得更模糊或刪除困難場景。

入場核對通過的範圍只有已知模擬條件。真實儀器校準、實際光暈／相位／材料響應、非柱體獨立細節答案、平台絕對位移與新物理樣品確認仍然欠缺。資料來源独立也不等於設計完全沒有受到既有開發影像啟發。真實顯微影像與 SEM 仍是有用性判斷的主體。

In [10]:
summary = {
    "audited_scenes": len(audit_v2["scenes"]), "audited_views": len(views_v2), "core_pixels_per_version": int(views_v2["count"].sum()),
    "metrics": metrics.to_dict(orient="records"), "hard_cases": hard_cases.to_dict(orient="records"),
    "categories": category_table.to_dict(orient="records"), "decomposition": decomposition_table.to_dict(orient="records"),
    "replay_patches": len(replays), "replay_pixels": int(replays["pixels"].sum()),
    "replay_maximum_gray": float(max(replays.noiseless_max_gray.max(), replays.target_max_gray.max())),
    "precision_v1_max_gray": precision["float64_local_replay_vs_stored"]["max_abs_gray"],
    "native_area_max_abs_v2": max(scene["target_area_integration"]["max_abs"] for scene in audit_v2["scenes"]),
    "same_physical_conditions": len(condition_rows), "v1_all_payloads_and_metadata_unchanged": integrity["all_v1_payloads_and_metadata_unchanged"],
    "v1_manifest_sha256": audit_v1["manifest_sha256"], "v2_manifest_sha256": audit_v2["manifest_sha256"],
    "generator_v2_sha256": audit_v2["source_generator_sha256"],
    "real_calibration_qualified": audit_v2["real_calibration_qualified"], "real_usefulness_established": audit_v2["real_usefulness_established"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "audited_scenes": 64,
  "audited_views": 128,
  "core_pixels_per_version": 2097152,
  "metrics": [
    {
      "version": "v1",
      "views": 128,
      "core_pixels": 2097152,
      "worst_RMS_over_sigma": 0.8176902759470385,
      "worst_p99_over_sigma": 2.047052107879339,
      "absolute_max_gray": 1.9862902164459229,
      "failed_operator_views": 6,
      "failed_operator_scenes": 3,
      "all_gate_failed_scenes": 4,
      "pilot_admitted": false
    },
    {
      "version": "v2",
      "views": 128,
      "core_pixels": 2097152,
      "worst_RMS_over_sigma": 0.16601320524597346,
      "worst_p99_over_sigma": 0.45747615149046433,
      "absolute_max_gray": 0.44565528631210327,
      "failed_operator_views": 0,
      "failed_operator_scenes": 0,
      "all_gate_failed_scenes": 0,
      "pilot_admitted": true
    }
  ],
  "hard_cases": [
    {
      "scene_id": "scene_007",
      "v1_rms_over_declared_sigma": 0.03975687113919175,
      "v1_p99_abs_over_declared_sigma": 0.2202